# NB 6.3 &mdash; Crash test

**MP 5149** &mdash; Desenvolupament de components software per a sistemes d'aprenentatge automàtic

**UT6** &mdash; Xarxes neuronals convolucionals i introducció a la visió per computador

---
### D'on venim

Al NB 6.2 vas buscar la xarxa amb més encert. Però l'encert només diu com es comporta una xarxa
amb imatges que s'assemblen a les de l'entrenament. **I al món real?** Una càmera bruta, una foto
moguda o una mica menys de llum.

### Què farem avui

Els cotxes es proven estavellant-los contra una paret. Avui farem el mateix amb dues xarxes ja
entrenades, sense esperar cap entrenament: els posarem soroll, les farem girar, els taparem
trossos i, al final, **les enganyarem a posta**. Tot sortirà en una **fitxa de resistència**
que tornaràs a fer servir al projecte del NB 6.6.

### Preguntes que has de saber respondre en acabar

1. Quines alteracions fan caure més una CNN? Coincideixen amb les que et molesten a tu?
2. La data augmentation fa que una xarxa ho aguanti tot?
3. Com es pot saber a quina part de la imatge mira una xarxa?
4. Què és un atac adversari, i per què cal tan poca força?
5. Què hauria de dir la fitxa d'una xarxa abans de posar-la en producció?

## 0. Preparació

Les biblioteques, la plantilla i les eines del NB 6.2.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch.utils.data import DataLoader
from torchvision.transforms import v2

plt.rcParams["figure.figsize"] = (6, 4)

La **plantilla d'entrenament**, igual que abans. No cal modificar-la.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Plantilla d'entrenament. No cal modificar-la: només cal saber què fa cada funció.
# Fa el mateix que MLPClassifier feia per dins a la UT5, però a la vista.
# ─────────────────────────────────────────────────────────────────────────────
import math
import time

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader

SEED = 42
LOSS_LIMIT = 100       # en un entrenament sa, la loss no s'hi acosta mai
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"   # GPU si n'hi ha; si no, CPU


def make_cnn(in_channels, n_classes, image_size=28, filters=(16, 32), pooling=True,
             global_pool=False, batch_norm=False, dropout=0.0):
    """Crea una CNN a partir de paràmetres, com MLPClassifier amb hidden_layer_sizes.

    Paràmetres:
        in_channels: canals de la imatge (1 en gris, 3 en color).
        n_classes: quantes classes ha de distingir.
        image_size: costat de la imatge en píxels.
        filters: filtres de cada capa convolucional; (16, 32) són dues capes.
        pooling: si cada capa redueix la imatge a la meitat amb MaxPool2d.
        global_pool: si al final es fa la mitjana de cada mapa en lloc d'aplanar-los.
        batch_norm: si s'afegeix BatchNorm2d després de cada convolució.
        dropout: fracció de neurones que s'apaguen a l'atzar a l'entrenament.

    Retorna el model. Exemple: model = make_cnn(1, 10, filters=(16, 32, 64))
    """
    if isinstance(filters, int):
        raise ValueError(f"filters ha de ser una llista de capes: per a una sola capa, escriu filters=({filters},), amb la coma.")
    if not len(filters) or min(filters) < 1:
        raise ValueError("filters necessita almenys una capa, amb un nombre positiu de filtres: per exemple, filters=(16, 32).")
    if not 0 <= dropout < 1:
        raise ValueError(f"dropout és la fracció de neurones que s'apaguen i ha d'anar de 0 a menys d'1: {dropout} no val. Prova dropout=0.5.")
    n_pools = (len(filters) - 1 if global_pool else len(filters)) if pooling else 0
    side = image_size // 2 ** n_pools
    if side < 1:
        raise ValueError(f"Hi ha massa capes amb pooling per a una imatge de {image_size} píxels: "
                         "fes filters més curt o prova pooling=False.")

    torch.manual_seed(SEED)
    layers = []
    channels = in_channels
    for position, n_filters in enumerate(filters):
        layers += [nn.Conv2d(channels, n_filters, kernel_size=3, padding=1)]
        if batch_norm:
            layers += [nn.BatchNorm2d(n_filters)]
        layers += [nn.ReLU()]
        is_last = position == len(filters) - 1
        if pooling and not (global_pool and is_last):
            layers += [nn.MaxPool2d(2)]
        channels = n_filters

    switch_off = [nn.Dropout(dropout)] if dropout > 0 else []
    if global_pool:
        layers += [nn.AdaptiveAvgPool2d(1), nn.Flatten(), *switch_off, nn.Linear(channels, n_classes)]
    else:
        layers += [nn.Flatten(), nn.Linear(channels * side * side, 64), nn.ReLU(),
                   *switch_off, nn.Linear(64, n_classes)]
    return nn.Sequential(*layers)


def count_parameters(model):
    """Retorna quants pesos entrenables té el model."""
    return sum(weights.numel() for weights in model.parameters() if weights.requires_grad)


def make_loaders(train_set, val_set, batch_size=64, augment=None):
    """Prepara els mini-batches d'entrenament i de validació.

    Paràmetres:
        train_set, val_set: conjunts de dades de PyTorch.
        batch_size: imatges per mini-batch.
        augment: transformació per a cada imatge d'entrenament (data augmentation) o None.

    Retorna (train_loader, val_loader). Només l'entrenament es barreja i s'augmenta.
    """
    def collate_augmented(batch):
        images = torch.stack([augment(image) for image, _ in batch])
        labels = torch.as_tensor([label for _, label in batch])
        return images, labels

    collate = collate_augmented if augment is not None else None
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, collate_fn=collate)
    val_loader = DataLoader(val_set, batch_size=1000)
    return train_loader, val_loader


@torch.no_grad()
def _evaluate(model, loader):
    """Retorna la loss mitjana i l'encert del model amb un loader."""
    model.eval()
    device = next(model.parameters()).device
    loss_function = nn.CrossEntropyLoss(reduction="sum")
    total_loss, hits, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        scores = model(images)
        total_loss += loss_function(scores, labels).item()
        hits += (scores.argmax(dim=1) == labels).sum().item()
        total += len(labels)
    return total_loss / total, hits / total


def accuracy(model, loader):
    """Retorna l'encert del model (de 0 a 1), com score() a scikit-learn."""
    return _evaluate(model, loader)[1]


def train_model(model, train_loader, val_loader, epochs, lr, optimizer="adam"):
    """Entrena el model i mostra com va, epoch a epoch. És el fit() d'aquesta unitat.

    Paràmetres:
        model: la xarxa, feta amb make_cnn.
        train_loader, val_loader: els mini-batches, fets amb make_loaders.
        epochs: quantes vegades es recorren totes les dades d'entrenament.
        lr: el learning rate.
        optimizer: "adam" o "sgd".

    Retorna un diccionari amb les llistes train_loss, val_loss, train_acc i val_acc.
    """
    optimizers = {"adam": torch.optim.Adam, "sgd": torch.optim.SGD}
    if optimizer not in optimizers:
        raise ValueError(f"Optimitzador desconegut: {optimizer!r}. Tria 'adam' o 'sgd'.")

    torch.manual_seed(SEED)
    check_data(model, val_loader)
    model.to(DEVICE)
    step_maker = optimizers[optimizer](model.parameters(), lr=lr)
    loss_function = nn.CrossEntropyLoss()
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(1, epochs + 1):
        start = time.perf_counter()
        model.train()                                  # activa dropout i batch norm
        total_loss, hits, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            step_maker.zero_grad()                     # esborra el repartiment anterior
            scores = model(images)                     # 1. la xarxa prediu
            _check_labels(scores, labels)
            loss = loss_function(scores, labels)       # 2. mesurem l'error (la loss)
            loss.backward()                            # 3. backpropagation: repartim la culpa
            step_maker.step()                          # 4. una passa de gradient descent
            total_loss += loss.item() * len(labels)
            hits += (scores.argmax(dim=1) == labels).sum().item()
            total += len(labels)

        val_loss, val_acc = _evaluate(model, val_loader)
        history["train_loss"].append(total_loss / total)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(hits / total)
        history["val_acc"].append(val_acc)
        print(f"Epoch {epoch}/{epochs} · loss {total_loss / total:.3f} · "
              f"encert a validació {val_acc:.1%} · {time.perf_counter() - start:.0f} s")

        if not math.isfinite(total_loss) or total_loss / total > LOSS_LIMIT:
            print("La loss s'ha disparat i l'entrenament s'atura: "
                  "segurament el learning rate és massa alt.")
            break
    return history


def check_data(model, loader):
    """Prova la xarxa amb una imatge del conjunt i, si no l'accepta, explica què cal canviar."""
    image, _ = loader.dataset[0]
    model.to(DEVICE)
    model.eval()
    try:
        with torch.no_grad():
            model(image.unsqueeze(0).to(DEVICE))
    except RuntimeError as error:
        channels, height, width = image.shape
        raise ValueError(f"La xarxa no accepta aquestes imatges, que tenen {channels} canals i fan {height} × {width} "
                         f"píxels. Crea-la de nou amb make_cnn({channels}, n_classes, image_size={height}), "
                         "amb la resta de paràmetres que ja tenies.") from error


def _check_labels(scores, labels):
    """Si hi ha més classes que sortides, la loss no es pot calcular: ho expliquem en lloc de fallar."""
    if labels.max() >= scores.shape[1]:
        raise ValueError(f"Aquest conjunt té etiquetes fins a la {labels.max().item()}, però la xarxa només té "
                         f"{scores.shape[1]} sortides. Crea-la de nou amb n_classes igual al nombre de classes: len(names).")


def plot_history(history):
    """Dibuixa les corbes de loss i d'encert d'un entrenament."""
    fig, grafics = plt.subplots(1, 2, figsize=(10, 3.5))
    epochs = range(1, len(history["train_loss"]) + 1)
    grafics[0].plot(epochs, history["train_loss"], "o-", label="entrenament")
    grafics[0].plot(epochs, history["val_loss"], "o-", label="validació")
    grafics[0].set_ylabel("loss")
    if not any(math.isfinite(value) for value in history["train_loss"] + history["val_loss"]):
        grafics[0].text(0.5, 0.5, "La loss s'ha disparat:\nprova un learning rate més petit.",
                        transform=grafics[0].transAxes, ha="center", va="center")
    grafics[1].plot(epochs, history["train_acc"], "o-", label="entrenament")
    grafics[1].plot(epochs, history["val_acc"], "o-", label="validació")
    grafics[1].set_ylabel("encert")
    for grafic in grafics:
        grafic.set_xlabel("epoch")
        grafic.legend()
    plt.tight_layout()
    plt.show()

Les eines del NB 6.2, com `load_dataset`, i les eines noves del crash test. Tampoc cal modificar-les.

In [ ]:
# Llenç per dibuixar a Colab. No cal modificar-la.
import base64
import io

import numpy as np
from IPython.display import HTML, display
from PIL import Image

CANVAS_PAGE = """
<canvas id="llenc" width="{size}" height="{size}"
        style="background:black;border-radius:6px;touch-action:none;cursor:crosshair"></canvas><br>
<button id="fet" style="margin-top:6px;padding:6px 18px">Fet!</button>
<script>
const canvas = document.getElementById("llenc");
const context = canvas.getContext("2d");
context.strokeStyle = "white"; context.lineWidth = {size} / 28 * 1.6;
context.lineCap = "round"; context.lineJoin = "round";
let drawing = false;
const position = e => {{ const r = canvas.getBoundingClientRect(); return [e.clientX - r.left, e.clientY - r.top]; }};
canvas.onpointerdown = e => {{ drawing = true; context.beginPath(); context.moveTo(...position(e)); }};
canvas.onpointermove = e => {{ if (drawing) {{ context.lineTo(...position(e)); context.stroke(); }} }};
window.onpointerup = () => {{ drawing = false; }};
window.waitForDrawing = () => new Promise(resolve => {{
  document.getElementById("fet").onclick = () => resolve(canvas.toDataURL("image/png"));
}});
</script>
"""


def bitmap_from_png(png):
    """Converteix el dibuix en una imatge de 28×28 centrada, com les de Quick Draw.

    Retorna un array uint8, o None si el llenç és buit.
    """
    image = np.array(Image.open(io.BytesIO(png)).convert("RGBA"))
    ink = image[:, :, :3].max(axis=2) * (image[:, :, 3] > 0)
    rows, cols = np.nonzero(ink > 30)
    if len(rows) == 0:
        print("El llenç és buit: dibuixa alguna cosa i torna a executar la cel·la.")
        return None
    cropped = Image.fromarray(ink[rows.min():rows.max() + 1, cols.min():cols.max() + 1].astype(np.uint8))
    scale = 24 / max(cropped.size)
    new_size = (max(1, round(cropped.width * scale)), max(1, round(cropped.height * scale)))
    bitmap = Image.new("L", (28, 28), 0)
    bitmap.paste(cropped.resize(new_size, Image.LANCZOS), ((28 - new_size[0]) // 2, (28 - new_size[1]) // 2))
    return np.array(bitmap, dtype=np.uint8)


def draw_canvas(size=280):
    """Obre un llenç, espera que premis «Fet!» i retorna el dibuix de 28×28.

    Només funciona a Colab. Retorna None fora de Colab o si el llenç és buit.
    """
    try:
        from google.colab import output
    except ImportError:
        print("El llenç només funciona a Colab. Aquí pots provar el model amb una imatge "
              "del conjunt de test, amb predict_top3.")
        return None
    display(HTML(CANVAS_PAGE.format(size=size)))
    data_url = output.eval_js("waitForDrawing()", timeout_sec=600)
    return bitmap_from_png(base64.b64decode(data_url.split(",", 1)[1]))


# Funcions del laboratori: carregar dades, jugar i comparar. No cal modificar-la.
import io
import tempfile
import unicodedata
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import TensorDataset

URL_BASE = "https://raw.githubusercontent.com/pprohenspolitecnicllevant/desenvolupament-components-software/refs/heads/main/"
URL_CIFAR = "https://huggingface.co/datasets/uoft-cs/cifar10/resolve/main/plain_text/"
CACHE = Path(tempfile.gettempdir()) / "dades_ut6"
CIFAR_NAMES = ["avió", "cotxe", "ocell", "gat", "cérvol", "gos", "granota", "cavall", "vaixell", "camió"]
NPZ_FILES = {"mnist": "UT01-Xarxes_neuronals_supervisades/mnist/mnist.npz",
             "fashion": "UT06-Xarxes_convolucionals/fashion_mnist/fashion_mnist.npz",
             "quickdraw": "UT06-Xarxes_convolucionals/quickdraw/quickdraw_30.npz"}
N_VAL = {"mnist": 10_000, "fashion": 10_000, "cifar10": 10_000, "quickdraw": 15_000}


def _download(url):
    """Baixa un fitxer només la primera vegada i retorna on és."""
    target = CACHE / url.rsplit("/", 1)[1]
    if not target.exists():
        CACHE.mkdir(parents=True, exist_ok=True)
        try:
            with urllib.request.urlopen(url, timeout=300) as response:
                content = response.read()
        except OSError as error:
            raise RuntimeError(f"No s'han pogut baixar les dades de {url}. Comprova la connexió "
                               f"i torna a executar la cel·la. Detall: {error}") from error
        partial = target.with_suffix(".part")
        partial.write_bytes(content)
        partial.rename(target)
    return target


def _read_cifar(part):
    """Llegeix una part de CIFAR-10 com a imatges (N, 32, 32, 3) i etiquetes."""
    table = pd.read_parquet(_download(f"{URL_CIFAR}{part}-00000-of-00001.parquet"))
    images = np.stack([np.array(Image.open(io.BytesIO(cell["bytes"]))) for cell in table["img"]])
    return images, table["label"].to_numpy()


def _read_arrays(name):
    """Retorna imatges (N, C, H, W) en uint8, etiquetes i noms de classe d'un conjunt."""
    if name == "cifar10":
        X_train, y_train = _read_cifar("train")
        X_test, y_test = _read_cifar("test")
        return X_train.transpose(0, 3, 1, 2), y_train, X_test.transpose(0, 3, 1, 2), y_test, CIFAR_NAMES
    with np.load(_download(URL_BASE + NPZ_FILES[name]), allow_pickle=False) as data:
        names = [str(n) for n in data["class_names"]] if "class_names" in data.files else [str(n) for n in range(10)]
        return (data["X_train"].reshape(-1, 1, 28, 28), data["y_train"],
                data["X_test"].reshape(-1, 1, 28, 28), data["y_test"], names)


def _to_dataset(images, labels):
    """Passa imatges en uint8 a un TensorDataset amb valors de 0 a 1."""
    return TensorDataset(torch.tensor(images / 255.0, dtype=torch.float32),
                         torch.tensor(labels.astype(int)))


def load_dataset(name, n_train=None, seed=42, n_val=None):
    """Carrega un dels quatre conjunts de la unitat, ja preparat per a una CNN.

    Paràmetres:
        name: "mnist", "fashion", "cifar10" o "quickdraw".
        n_train: quantes imatges d'entrenament es volen, triades a l'atzar. None, totes.
        seed: la llavor d'aquest sorteig.
        n_val: quantes imatges de validació es volen. None, totes. Si en són menys, va més de pressa.

    Retorna (train_set, val_set, test_set, class_names).
    """
    if name not in N_VAL:
        raise ValueError(f"Conjunt desconegut: {name!r}. Tria'n un d'aquests: {', '.join(N_VAL)}.")
    X_train, y_train, X_test, y_test, names = _read_arrays(name)
    n_fit = len(X_train) - N_VAL[name]
    if n_train is not None and not 1 <= n_train <= n_fit:
        raise ValueError(f"{name} només té {n_fit} imatges d'entrenament: tria un n_train entre 1 i {n_fit}.")
    chosen = np.arange(n_fit) if n_train is None else np.random.default_rng(seed).permutation(n_fit)[:n_train]
    stop = None if n_val is None else n_fit + n_val
    return (_to_dataset(X_train[chosen], y_train[chosen]), _to_dataset(X_train[n_fit:stop], y_train[n_fit:stop]),
            _to_dataset(X_test, y_test), names)


def _as_picture(image):
    """Prepara un tensor (C, H, W) per dibuixar-lo amb imshow."""
    return image.permute(1, 2, 0).squeeze().cpu().numpy()


def show_examples(dataset, class_names, n=10):
    """Dibuixa les primeres imatges d'un conjunt amb el nom de la seva classe."""
    fig, grafics = plt.subplots(1, n, figsize=(1.2 * n, 1.8))
    for number, grafic in enumerate(grafics):
        image, label = dataset[number]
        grafic.imshow(_as_picture(image), cmap="gray")
        grafic.set_title(class_names[int(label)], fontsize=9)
        grafic.axis("off")
    plt.show()


def show_augmented(image, augment, n=8):
    """Dibuixa la imatge original i n versions que en fa la data augmentation."""
    fig, grafics = plt.subplots(1, n + 1, figsize=(1.2 * (n + 1), 1.8))
    grafics[0].imshow(_as_picture(image), cmap="gray")
    grafics[0].set_title("original", fontsize=9)
    for grafic in grafics[1:]:
        grafic.imshow(_as_picture(augment(image)), cmap="gray")
    for grafic in grafics:
        grafic.axis("off")
    plt.show()


@torch.no_grad()
def predict_top3(model, image, class_names):
    """Retorna les tres classes més probables d'una imatge: [(nom, probabilitat), ...]."""
    model.eval()
    device = next(model.parameters()).device
    probabilities = torch.softmax(model(image.unsqueeze(0).to(device))[0], dim=0).cpu()
    values, positions = probabilities.topk(3)
    return [(class_names[position], value) for position, value in zip(positions.tolist(), values.tolist())]


def keep_classes(dataset, class_names, chosen):
    """Es queda amb les classes triades i les renumera des de 0.

    Retorna (conjunt reduït, llista de noms en l'ordre de chosen).
    """
    for name in chosen:
        if name not in class_names:
            raise ValueError(f"La classe {name!r} no hi és. Les que hi ha: {', '.join(class_names)}.")
    images, labels = dataset.tensors
    new_labels = torch.full_like(labels, -1)
    for number, name in enumerate(chosen):
        new_labels[labels == class_names.index(name)] = number
    keep = new_labels >= 0
    return TensorDataset(images[keep], new_labels[keep]), list(chosen)


def pictionary(model, class_names):
    """Obre el llenç, passa el dibuix pel model i mostra les tres classes més probables.

    Retorna el top-3, o None si no hi ha dibuix (llenç buit o fora de Colab).
    """
    drawing = draw_canvas()
    if drawing is None:
        return None
    image = torch.tensor(drawing / 255.0, dtype=torch.float32).unsqueeze(0)
    top = predict_top3(model, image, class_names)
    fig, grafics = plt.subplots(1, 2, figsize=(7, 2.5))
    grafics[0].imshow(drawing, cmap="gray")
    grafics[0].set_title("el que veu la xarxa")
    grafics[0].axis("off")
    grafics[1].barh([name for name, _ in top][::-1], [value for _, value in top][::-1], color="#1C7293")
    grafics[1].set_xlim(0, 1)
    grafics[1].set_xlabel("probabilitat")
    plt.tight_layout()
    plt.show()
    return top


def make_quiz(dataset, n=20, seed=7):
    """Tria sempre les mateixes n imatges d'un conjunt per al qüestionari."""
    chosen = np.random.default_rng(seed).permutation(len(dataset))[:n]
    images, labels = dataset.tensors
    return TensorDataset(images[chosen], labels[chosen])


def show_quiz(quiz, class_names):
    """Dibuixa les imatges del qüestionari numerades, sense dir-ne la classe."""
    print("Classes possibles:", ", ".join(class_names))
    fig, grafics = plt.subplots(4, 5, figsize=(9, 7.5))
    for number, grafic in enumerate(grafics.flat):
        grafic.imshow(_as_picture(quiz[number][0]), cmap="gray", interpolation="nearest")
        grafic.set_title(number + 1)
        grafic.axis("off")
    plt.show()


def _plain(text):
    """Treu majúscules, accents i espais dels extrems per comparar respostes."""
    return "".join(letter for letter in unicodedata.normalize("NFD", text.strip().lower())
                   if not unicodedata.combining(letter))


@torch.no_grad()
def quiz_results(model, quiz, answers, class_names):
    """Compara les respostes d'una persona i les del model amb la classe real.

    Retorna una taula amb les columnes real, persona i CNN, i mostra el marcador.
    """
    images, labels = quiz.tensors
    if len(answers) != len(images):
        raise ValueError(f"Calen {len(images)} respostes i n'hi ha {len(answers)}.")
    known = {_plain(name): name for name in class_names}
    unknown = [answer for answer in answers if _plain(answer) not in known]
    if unknown:
        raise ValueError(f"Resposta desconeguda: {unknown[0]!r}. Les classes són: {', '.join(class_names)}.")
    model.eval()
    device = next(model.parameters()).device
    guesses = model(images.to(device)).argmax(dim=1).cpu().tolist()
    table = pd.DataFrame({"real": [class_names[label] for label in labels.tolist()],
                          "persona": [known[_plain(answer)] for answer in answers],
                          "CNN": [class_names[guess] for guess in guesses]},
                         index=range(1, len(images) + 1))
    print(f"Persona: {(table['persona'] == table['real']).sum()} de {len(table)} · "
          f"CNN: {(table['CNN'] == table['real']).sum()} de {len(table)}")
    return table


# Funcions per posar a prova una xarxa. No cal modificar-la.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torchvision.transforms import v2

DEFAULT_LEVELS = {"noise": [0, 0.05, 0.1, 0.2, 0.3],        # desviació del soroll (píxels de 0 a 1)
                  "blur": [0, 0.5, 1.0, 1.5, 2.0],           # força del difuminat
                  "rotation": [0, 10, 20, 30, 45],           # graus
                  "brightness": [0, 0.1, 0.2, 0.3, 0.5],     # quant s'aclareix
                  "occlusion": [0, 0.125, 0.25, 0.375, 0.5]}  # costat del tros tapat, en fracció de la imatge
CRASH_NAMES = {"noise": "soroll", "blur": "difuminat", "rotation": "gir",
               "brightness": "brillantor", "occlusion": "tros tapat"}


def load_model(name):
    """Carrega una de les dues CNN de CIFAR-10 ja entrenades: "cifar_cnn" o "cifar_cnn_aug"."""
    if name not in ("cifar_cnn", "cifar_cnn_aug"):
        raise ValueError(f"Model desconegut: {name!r}. Els que hi ha: cifar_cnn, cifar_cnn_aug.")
    model = make_cnn(3, 10, image_size=32, filters=(16, 32, 64), batch_norm=True)
    file = _download(f"{URL_BASE}UT06-Xarxes_convolucionals/models/{name}.pt")
    model.load_state_dict(torch.load(file, weights_only=True, map_location="cpu"))
    return model.eval()


def corrupt(images, kind, level, generator):
    """Espatlla un lot d'imatges amb la prova i la intensitat indicades."""
    if level == 0:
        return images
    if kind == "noise":
        return (images + level * torch.randn(images.shape, generator=generator)).clamp(0, 1)
    if kind == "blur":
        return v2.functional.gaussian_blur(images, kernel_size=2 * int(3 * level) + 1, sigma=float(level))
    if kind == "rotation":
        return v2.functional.rotate(images, float(level))
    if kind == "brightness":
        return (images + level).clamp(0, 1)
    side = images.shape[-1]
    patch = max(1, round(level * side))
    tops = torch.randint(0, side - patch + 1, (len(images),), generator=generator).tolist()
    lefts = torch.randint(0, side - patch + 1, (len(images),), generator=generator).tolist()
    covered = images.clone()
    for position, (top, left) in enumerate(zip(tops, lefts)):
        covered[position, :, top:top + patch, left:left + patch] = 0.5
    return covered


@torch.no_grad()
def crash_test(model, loader, kind, levels=None):
    """Mesura l'encert del model mentre s'espatllen les imatges cada vegada més.

    Paràmetres:
        model: la xarxa que es prova.
        loader: les imatges de prova.
        kind: "noise", "blur", "rotation", "brightness" o "occlusion".
        levels: les intensitats. Si no se'n donen, es fan servir unes per defecte.

    Retorna una taula amb les columnes intensitat i encert.
    """
    if kind not in DEFAULT_LEVELS:
        raise ValueError(f"Prova desconeguda: {kind!r}. Tria'n una d'aquestes: {', '.join(DEFAULT_LEVELS)}.")
    model.eval()
    device = next(model.parameters()).device
    rows = []
    for level in (DEFAULT_LEVELS[kind] if levels is None else levels):
        generator = torch.Generator().manual_seed(SEED)
        hits, total = 0, 0
        for images, labels in loader:
            guesses = model(corrupt(images, kind, level, generator).to(device)).argmax(dim=1).cpu()
            hits += (guesses == labels).sum().item()
            total += len(labels)
        rows.append({"intensitat": level, "encert": hits / total})
    return pd.DataFrame(rows)


def plot_crash(tables, title=""):
    """Dibuixa corbes de resistència. tables és un diccionari {nom del model: taula de crash_test}."""
    fig, grafic = plt.subplots()
    for label, table in tables.items():
        grafic.plot(table["intensitat"], table["encert"], "o-", label=label)
    grafic.set_xlabel("intensitat")
    grafic.set_ylabel("encert")
    grafic.set_ylim(0, 1)
    grafic.set_title(title)
    grafic.legend()
    plt.show()
    return grafic


def resistance_card(model, loader):
    """Fa totes les proves i les resumeix en una fitxa de resistència."""
    rows = {}
    for kind, name in CRASH_NAMES.items():
        table = crash_test(model, loader, kind)
        clean, worst = table["encert"].iloc[0], table["encert"].iloc[-1]
        rows[name] = {"encert sense tocar": round(clean, 3),
                      "encert a la intensitat més forta": round(worst, 3),
                      "pèrdua (punts)": round(100 * (clean - worst), 1)}
    return pd.DataFrame(rows).T


def show_corruptions(image, kind):
    """Dibuixa una imatge amb totes les intensitats per defecte d'una prova."""
    levels = DEFAULT_LEVELS[kind]
    fig, grafics = plt.subplots(1, len(levels), figsize=(1.5 * len(levels), 1.9))
    generator = torch.Generator().manual_seed(SEED)
    for level, grafic in zip(levels, grafics):
        changed = corrupt(image.unsqueeze(0), kind, level, generator)[0]
        grafic.imshow(changed.permute(1, 2, 0).squeeze().numpy(), cmap="gray")
        grafic.set_title(str(level), fontsize=9)
        grafic.axis("off")
    fig.suptitle(f"{CRASH_NAMES[kind]}: intensitats", y=1.02)
    plt.show()


@torch.no_grad()
def occlusion_map(model, image, patch=6):
    """Tapa cada zona de la imatge i mesura quant baixa la confiança en la classe predita.

    Retorna una graella: com més alt el valor, més importava aquella zona per decidir.
    """
    model.eval()
    device = next(model.parameters()).device
    image = image.to(device)
    probabilities = torch.softmax(model(image.unsqueeze(0))[0], dim=0)
    chosen = probabilities.argmax()
    fill = image.mean(dim=(1, 2), keepdim=True)
    starts = range(0, image.shape[-1] - patch + 1, 2)
    result = np.zeros((len(starts), len(starts)))
    for row, top in enumerate(starts):
        covered = image.repeat(len(starts), 1, 1, 1)
        for col, left in enumerate(starts):
            covered[col, :, top:top + patch, left:left + patch] = fill
        result[row] = (probabilities[chosen] - torch.softmax(model(covered), dim=1)[:, chosen]).cpu().numpy()
    return result


def show_occlusion(model, image, class_names, patch=6):
    """Dibuixa la imatge i el seu mapa d'oclusió, amb la classe predita."""
    result = occlusion_map(model, image, patch)
    model.eval()
    with torch.no_grad():
        scores = model(image.unsqueeze(0).to(next(model.parameters()).device))[0]
    picture = image.permute(1, 2, 0).squeeze().cpu().numpy()
    fig, grafics = plt.subplots(1, 2, figsize=(6.5, 3))
    grafics[0].imshow(picture, cmap="gray")
    grafics[0].set_title(f"diu: {class_names[int(scores.argmax())]}")
    grafics[1].imshow(picture, cmap="gray", alpha=0.35)
    side = image.shape[-1]
    limit = max(abs(result).max(), 1e-6)
    grafics[1].imshow(result, cmap="RdBu_r", vmin=-limit, vmax=limit, alpha=0.75, extent=(0, side, side, 0))
    grafics[1].set_title("vermell: on mira")
    for grafic in grafics:
        grafic.axis("off")
    plt.show()


def attack(model, image, force):
    """Canvia cada píxel, com a molt, force, en la direcció que més fa dubtar el model.

    És la backpropagation al revés: en lloc de canviar els pesos per reduir la loss,
    es canvien els píxels per augmentar-la. Retorna la imatge atacada.
    """
    if force == 0:
        return image.clone()
    model.eval()
    device = next(model.parameters()).device
    with torch.enable_grad():
        pixels = image.to(device).unsqueeze(0).clone().requires_grad_(True)
        scores = model(pixels)
        loss = nn.functional.cross_entropy(scores, scores.argmax(dim=1))
        direction, = torch.autograd.grad(loss, pixels)
    return (pixels + force * direction.sign()).clamp(0, 1)[0].detach().cpu()


def attacked_accuracy(model, dataset, force, n=200):
    """Retorna l'encert del model amb les primeres n imatges, atacades una a una."""
    model.eval()
    device = next(model.parameters()).device
    hits = 0
    for position in range(min(n, len(dataset))):
        image, label = dataset[position]
        with torch.no_grad():
            guess = model(attack(model, image, force).unsqueeze(0).to(device)).argmax(dim=1).item()
        hits += guess == int(label)
    return hits / min(n, len(dataset))


@torch.no_grad()
def show_attack(model, image, force, class_names):
    """Dibuixa l'original, el canvi (molt augmentat) i la imatge atacada, amb què diu el model."""
    attacked = attack(model, image, force)
    device = next(model.parameters()).device
    model.eval()
    fig, grafics = plt.subplots(1, 3, figsize=(7.5, 2.9))
    for picture, grafic, label in [(image, grafics[0], "original"), (attacked, grafics[2], "atacada")]:
        probabilities = torch.softmax(model(picture.unsqueeze(0).to(device))[0], dim=0)
        grafic.imshow(picture.permute(1, 2, 0).squeeze().cpu().numpy(), cmap="gray")
        grafic.set_title(f"{label}: {class_names[int(probabilities.argmax())]} ({probabilities.max().item():.0%})", fontsize=10)
    change = (attacked - image.cpu()).permute(1, 2, 0).squeeze().numpy()
    limit = max(abs(change).max(), 1e-6)
    grafics[1].imshow((change / limit + 1) / 2, cmap="gray")
    grafics[1].set_title(f"el canvi, augmentat (força {force})", fontsize=10)
    for grafic in grafics:
        grafic.axis("off")
    plt.show()

## 1. Els dos cotxes

Carreguem CIFAR-10 (Krizhevsky, 2009) i dues xarxes ja entrenades. Les proves es faran amb 2.000 imatges de validació,
que n'hi ha prou i va de pressa.

Les dues xarxes són germanes: `make_cnn(3, 10, image_size=32, filters=(16, 32, 64), batch_norm=True)`,
amb **90.058 pesos**, 40.000 imatges i 25 epochs. Només canvia una cosa:

| Model | Com s'ha entrenat |
|---|---|
| `clean_model` | Amb les imatges tal com són |
| `aug_model` | Amb data augmentation: mirall i desplaçament, com al NB 6.2 |

#### Un incís: carregar un model

Un fitxer `.pt` guarda **els pesos**: uns quants números per capa. Per fer-lo servir cal crear la mateixa
arquitectura i omplir-la. Això ho fa `load_model`, amb `torch.load(..., weights_only=True)`.

**Per què `weights_only=True`?** Perquè un fitxer `.pt` pot contenir també codi que s'executa en carregar-lo.
Amb aquesta opció només se'n llegeixen els números. **Fes-la servir sempre amb fitxers baixats d'internet.**

In [ ]:
train_set, val_set, _, names = load_dataset("cifar10", n_train=10_000, n_val=2_000)
loader = DataLoader(val_set, batch_size=500)

clean_model = load_model("cifar_cnn")
aug_model = load_model("cifar_cnn_aug")

print(f"Sense augmentation: {accuracy(clean_model, loader):.1%}")
print(f"Amb augmentation  : {accuracy(aug_model, loader):.1%}")

**69,8% i 78,4%.** Compara-ho amb el 50–60% de la competició del NB 6.2: amb 40.000 imatges i 25 epochs,
la mateixa mena de xarxa arriba molt més enllà. I aquí, amb tantes epochs, la data augmentation sí que
es nota. Però aquesta **no és la pregunta d'avui**.

## 2. Corbes de resistència

La funció `crash_test(model, loader, kind)` espatlla totes les imatges cada vegada més i mesura l'encert.
Hi ha cinc proves:

| `kind` | Què imita | Intensitats per defecte |
|---|---|---|
| `"noise"` | Una càmera dolenta o amb poca llum | Soroll de 0 a 0,3 |
| `"blur"` | Una foto moguda o desenfocada | Difuminat de 0 a 2 |
| `"rotation"` | Una càmera torta | De 0 a 45 graus |
| `"brightness"` | Massa llum | S'aclareix de 0 a 0,5 |
| `"occlusion"` | Un objecte al davant | Un tros tapat, fins a la meitat del costat |

Mirem com es veuen. La funció `show_corruptions(image, kind)` dibuixa una imatge a cada intensitat.

In [ ]:
show_corruptions(val_set[3][0], "noise")
show_corruptions(val_set[3][0], "occlusion")

I ara, la primera prova. `crash_test` retorna una taula, i `plot_crash` en dibuixa la corba.

In [ ]:
noise_table = crash_test(clean_model, loader, "noise")
plot_crash({"sense augmentation": noise_table}, "soroll")
noise_table.round(3)

Mira la imatge d'abans amb un soroll de 0,05: a tu no et costaria gens reconèixer-la. **La xarxa passa
del 69,8% al 55,6%.** Amb 0,3 es queda en un 13,4%, pràcticament l'atzar d'un 10%.

### Ara tu · Missió 1: la paret

1. **Aposta.** Ordena les cinc proves de la que fa més mal a la que en fa menys. Pensa en com ho faries
   tu: quina et costaria més?
2. **Experimenta.** Llança les cinc proves contra `clean_model`. Pots recórrer-les amb un `for` sobre
   `["noise", "blur", "rotation", "brightness", "occlusion"]`, i dibuixar cadascuna amb `plot_crash`.
3. **Compara** l'encert de totes a la intensitat més forta en una taula. Les intensitats no són
   comparables entre proves, però et donen una idea. Fixa't també en la intensitat més suau de
   cadascuna: quina fa caure més la xarxa amb un canvi que tu gairebé no veus?
4. **Explica.** Coincideix amb la teva aposta? Hi ha alguna prova que a tu no et molesti i a la xarxa sí?
   Recorda el que vas veure al NB 6.1 sobre els filtres: què busquen, i què els fa el soroll?

In [ ]:
# El teu codi aquí

## 3. Qui aguanta més?

`aug_model` ha vist imatges en mirall i desplaçades. **Això el fa més resistent a tot?**
`plot_crash` pot dibuixar diverses corbes alhora: `plot_crash({"sense": table_a, "amb": table_b}, "soroll")`.

### Ara tu · Missió 2: amb augmentation o sense

1. **Aposta.** A quines de les cinc proves creus que `aug_model` aguantarà millor? N'hi ha alguna en què
   creus que no servirà de res?
2. **Experimenta.** Dibuixa les dues corbes de cada prova en el mateix gràfic.
3. **Mesura quant perd cadascun**, no només quant encerta: `aug_model` ja parteix de més amunt. Fes una taula
   amb els punts perduts a la intensitat més forta de cada prova.
4. **Explica.** Hi ha alguna relació entre el que ha vist a l'entrenament (mirall i desplaçament) i el que
   aguanta millor? Què hauries de fer per a una càmera amb molt de soroll?

In [ ]:
# El teu codi aquí

**Què cal que t'enduguis.** Compara el resultat amb la teva aposta. La data augmentation d'aquest model fa
servir mirall i desplaçament, i cap imatge amb soroll. Una xarxa aguanta sobretot allò que ha vist. **Per això el crash test s'ha de
fer amb les alteracions que trobarà el model en producció**, no amb les que siguin còmodes de provar.

## 4. On mira?

Quan una xarxa encerta, potser ho fa per un motiu que no és el que creus. Un exemple que es va fer famós:
un model que distingia llops de gossos husky mirant si hi havia neu al fons. **Com ho podem saber?**

Amb una idea molt simple. Es tapa un tros de la imatge amb un quadrat gris i es mira quant baixa la
confiança de la xarxa en la seva resposta. Es repeteix per tota la imatge i queda un **mapa d'oclusió**.

| Què es tapa | Confiança en «gat» | Importància d'aquella zona |
|---|---|---|
| Res | 80% | |
| Una orella | 30% | **50 punts**: mirava aquí |
| Un tros del fons | 78% | 2 punts: no li importa |

Al mapa, el **vermell** és on, si tapes, la confiança baixa. El blau és on, si tapes, puja: aquella
zona li feia nosa. Fa servir la funció `show_occlusion(model, image, class_names)`.

In [ ]:
show_occlusion(aug_model, val_set[3][0], names)
print("Classe real:", names[int(val_set[3][1])])

### Ara tu · Missió 3: el detectiu

1. **Aposta.** Per a una foto d'un avió, on creus que mirarà? Tota la forma, les ales, o el cel blau?
2. **Experimenta.** Dibuixa el mapa de sis imatges diferents. Per trobar imatges d'una classe:
   `val_set.tensors[0][val_set.tensors[1] == names.index("avió")][0]`. Prova almenys dues classes d'animals
   i dues de vehicles.
3. **Compara** el mapa de `clean_model` i el de `aug_model` per a la mateixa imatge.
4. **Busca un error.** Troba una imatge en què la xarxa s'equivoqui i dibuixa'n el mapa. T'ajuda a entendre per què?
5. **Explica.** Mira l'objecte o el fons? Si una xarxa encerta mirant el cel, per què és un problema, si
   encerta igual?

In [ ]:
# El teu codi aquí

## 5. Enganyar-la

Fins ara hem espatllat les imatges a l'atzar. **I si les espatlléssim de la pitjor manera possible?**

Recorda la UT5: la backpropagation calcula com s'ha de moure cada pes perquè la loss baixi.
Es pot fer el mateix **amb els píxels i al revés**: calcular com s'ha de moure cada píxel perquè la loss
**pugi**. Després es mou cada píxel una quantitat petitíssima en aquesta direcció. És un **atac adversari**.

La funció `attack(model, image, force)` ho fa per tu. La força és el màxim que canvia un píxel:

| Força | Quant canvia cada píxel, de 0 a 255 |
|---|---|
| 0,005 | Poc més d'1 |
| 0,01 | Uns 2,5 |
| 0,05 | Uns 13 |

`show_attack` mostra la imatge original, el canvi (molt augmentat perquè es vegi) i la imatge atacada.

In [ ]:
show_attack(aug_model, val_set[3][0], 0.005, names)

**Un cérvol amb un 85% de confiança passa a ser una granota amb un 99%**, canviant cada píxel poc més d'1
sobre 255. Les dues imatges et semblaran iguals. La xarxa no s'ha espatllat: és que cada píxel, sumat a tots
els altres, l'empeny en la direcció que la fa fallar.

Compte: aquest atac sap exactament com és la xarxa per dins. Provar atacs només té sentit amb
xarxes pròpies o quan tens permís: és la manera de saber què no aguanten.

### Ara tu · Missió 4: quanta força cal?

1. **Aposta.** Amb quina força creus que l'encert baixarà a la meitat? Amb quina es començarà a notar a simple vista?
2. **Experimenta.** Amb `attacked_accuracy(model, val_set, force)`, que ataca les 200 primeres imatges,
   omple una taula amb les forces 0, 0,002, 0,005, 0,01, 0,02 i 0,05 per als dos models.
3. **Mira-ho.** Fes servir `show_attack` amb tres forces sobre una imatge que triïs. A partir de quina ja es nota?
4. **Explica.** Quin dels dos models aguanta millor l'atac? Ho esperaves? Què té a veure amb el soroll
   a l'atzar de la secció 2: per què fa molt més mal aquest canvi amb molt menys soroll?
   Pensa en un senyal de stop amb uns adhesius: quin problema tindria un cotxe autònom?

In [ ]:
# El teu codi aquí

## 6. La fitxa de resistència

Hora de resumir-ho. La funció `resistance_card(model, loader)` fa totes les proves i en treu una fitxa.

In [ ]:
resistance_card(aug_model, loader)

### Ara tu · Missió 5: la fitxa del teu model

1. **Entrena** la teva millor xarxa del NB 6.2 amb les mateixes regles: `train_set`, màxim 150.000 pesos i 5 epochs.
   Si no te'n recordes, prova `make_cnn(3, 10, image_size=32, filters=(16, 32, 64), batch_norm=True)`.
2. **Aposta.** Quina prova la farà caure més? Perdrà més o menys punts que `aug_model`?
3. **Fes-ne la fitxa** amb `resistance_card` i compara-la amb la de `aug_model`.
4. **Escriu per a un client,** en tres frases i sense paraules tècniques, què pot fer el teu model i en
   quines condicions no te'n pots refiar.

> ⏱ L'entrenament tarda menys d'un minut a un ordinador normal.

In [ ]:
# El teu codi aquí

## 7. Resum

| Pregunta | Resposta |
|---|---|
| **Què la fa caure?** | Amb CIFAR-10, sobretot el soroll i el gir: canvis que a tu no et molesten |
| **La data augmentation ho arregla?** | Només en part: una xarxa aguanta el que ha vist |
| **On mira?** | Els mapes d'oclusió ho mostren, i de vegades mira el fons |
| **La podem enganyar?** | Sí, amb canvis invisibles, si en coneixem l'interior |
| **Què cal abans de posar-la en producció?** | Una fitxa de resistència, amb les condicions reals |

### Les idees que t'has d'endur

1. **L'encert és només el principi.** Dos models amb el mateix encert poden aguantar coses molt diferents.
2. **Una xarxa no veu com tu.** El que a tu no et molesta la pot fer caure, i a l'inrevés.
3. **Prova el model amb el que es trobarà**, no amb el que sigui còmode.

## Exercicis finals

Les cinc missions les has fet pel camí. Aquests dos exercicis et demanen tot el notebook.

**Exercici 6. La vacuna.** Si una xarxa aguanta el que ha vist, entrena'n una que vegi soroll.
Amb `v2.GaussianNoise(sigma=0.05)` com a `augment` a `make_loaders`, entrena una xarxa de 90.058
pesos amb `train_set` i 5 epochs. Compara-la amb la mateixa xarxa sense vacuna a la prova `"noise"`. Quant ha guanyat?
Què ha perdut a les imatges netes? Li ha servit per a alguna altra prova?

**Exercici 7. Els dígits, a la paret.** Al NB 6.1 la CNN arribava al 98,5% amb MNIST. Entrena-la 3 epochs
amb `load_dataset("mnist")`, i fes-li la fitxa. MNIST és tan robust com semblava? En quina prova cau més?
(Compte: `make_cnn(1, 10)`, que MNIST és en gris.)

## Per al debat de classe

1. **Responsabilitat.** Si un cotxe autònom no veu un senyal perquè té un adhesiu, de qui és la culpa: de qui ha
   posat l'adhesiu, de qui ha entrenat la xarxa o de qui la va decidir fer servir?
2. **Explicacions.** El mapa d'oclusió diu on mira, però no per què. Seria suficient per justificar una decisió
   mèdica o judicial?
3. **Assegurar-se.** Quines proves demanaries a una empresa que et ven una càmera amb IA per a l'entrada de
   l'institut? Pensa què faries servir de la fitxa.